# Eksperimen SMSML - Breast Cancer Wisconsin Diagnostic

Notebook ini mengikuti alur eksperimen manual: data loading, EDA, preprocessing, split, dan penyimpanan dataset siap latih.

In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

RAW_PATH = Path('../breast_cancer_raw/wdbc.data')
OUTPUT_DIR = Path('breast_cancer_preprocessing')
OUTPUT_DIR.mkdir(exist_ok=True)

## Data Loading

In [ ]:
feature_groups = ('mean', 'se', 'worst')
base_features = (
    'radius', 'texture', 'perimeter', 'area', 'smoothness',
    'compactness', 'concavity', 'concave_points', 'symmetry', 'fractal_dimension'
)
feature_columns = [f'{name}_{group}' for group in feature_groups for name in base_features]
columns = ['id', 'diagnosis', *feature_columns]

df = pd.read_csv(RAW_PATH, header=None, names=columns)
df.head()

## Exploratory Data Analysis

In [ ]:
print('Shape:', df.shape)
display(df.info())
display(df.describe().T.head(10))
display(df['diagnosis'].value_counts())
print('Missing values:', int(df.isna().sum().sum()))
print('Duplicated rows:', int(df.duplicated().sum()))

In [ ]:
df['diagnosis'].value_counts().sort_index().plot(kind='bar')
plt.title('Diagnosis Distribution')
plt.show()

corr = df.drop(columns=['id', 'diagnosis']).corr()
plt.figure(figsize=(10, 8))
plt.imshow(corr, cmap='coolwarm', aspect='auto')
plt.colorbar()
plt.title('Feature Correlation')
plt.show()

## Preprocessing

In [ ]:
processed_df = df.drop(columns=['id']).drop_duplicates().copy()
processed_df['target'] = processed_df['diagnosis'].map({'B': 0, 'M': 1}).astype(int)
processed_df = processed_df.drop(columns=['diagnosis'])

x_columns = [col for col in processed_df.columns if col != 'target']
scaler = StandardScaler()
processed_df[x_columns] = scaler.fit_transform(processed_df[x_columns])

processed_df.head()

In [ ]:
train_df, test_df = train_test_split(
    processed_df,
    test_size=0.2,
    random_state=42,
    stratify=processed_df['target'],
)

processed_df.to_csv(OUTPUT_DIR / 'breast_cancer_processed.csv', index=False)
train_df.to_csv(OUTPUT_DIR / 'train.csv', index=False)
test_df.to_csv(OUTPUT_DIR / 'test.csv', index=False)

print('Full:', processed_df.shape)
print('Train:', train_df.shape)
print('Test:', test_df.shape)